# AI Clinical Observation System - Experiment Report

This notebook records the dataset design, model experiments, results, observed failure modes, and the next architecture direction for the AI Clinical Observation System.

The project goal is not to replace staff. The goal is to support human-in-the-loop observation by detecting safety-relevant visual cues, generating alarms for review, and drafting observation notes for staff approval.

## 1. Current System Summary

Current implemented pipeline:

1. Dataset of simulated clinical-observation videos under `dataset/raw/`.
2. Video-level labels in `dataset/training_manifest.jsonl`.
3. Baseline OpenCV nearest-centroid classifier.
4. Pretrained TorchVision `r3d_18` video action classifier fine-tuning path.
5. Grouped risk-label classifier for broader risk categories.
6. YOLO object detection scaffold for dangerous object cues.
7. MediaPipe pose/movement analyzer scaffold.
8. Live camera script using the combined clinical pipeline.
9. Backend prediction endpoint that can use the combined pipeline.

The most important conclusion so far: raw video fine-tuning alone is not strong enough with the current dataset size. The recommended direction is a hybrid system: YOLO + MediaPipe + rules + smaller classifier + LLM observation note generation.

In [ ]:
from pathlib import Path
import json
from collections import Counter

PROJECT_ROOT = Path.cwd()
MANIFEST_PATH = PROJECT_ROOT / "dataset" / "training_manifest.jsonl"

rows = [json.loads(line) for line in MANIFEST_PATH.open(encoding="utf-8") if line.strip()]
print("Dataset size:", len(rows))
print("Detailed labels:", len({row["label"] for row in rows}))
Counter(row["label"] for row in rows)

## 2. Dataset Summary

Current dataset snapshot:

- Total videos: **201**
- Detailed labels: **18**
- Labels include normal activity, movement/agitation, medical safety risks, violence/property risk, and object/self-harm risk placeholders.

Detailed label counts observed:

| Label | Count |
|---|---:|
| bleeding_visible | 21 |
| sharp_object_detected | 20 |
| sitting | 19 |
| sleeping | 17 |
| reading | 13 |
| head_banging | 12 |
| attack_on_person | 11 |
| fall | 11 |
| vomiting | 11 |
| aggressive_movement | 9 |
| eating | 8 |
| ligature_risk | 8 |
| property_damage | 8 |
| walking | 8 |
| choking_simulation | 7 |
| pacing | 6 |
| prolonged_inactivity | 6 |
| standing | 6 |

The key limitation is that many classes have fewer than 10 videos. That is too small for a strong 18-label video classifier.

In [ ]:
from ml.training.label_groups import grouped_label

group_counts = Counter(grouped_label(row["label"]) for row in rows)
group_counts

## 3. Grouped Risk Labels

Because 18 labels are too sparse, labels were grouped into broader categories:

| Group | Count |
|---|---:|
| normal_activity | 71 |
| object_self_harm_risk | 49 |
| medical_safety_risk | 35 |
| violence_property_risk | 31 |
| movement_agitation | 15 |

These grouped labels are more realistic for the current data volume. The grouped classifier is used as a risk-category classifier, while detailed alarms should increasingly rely on YOLO, MediaPipe, and rules.

## 4. Baseline Classifier Result

Baseline model:

- Method: OpenCV frame statistics + nearest-centroid classifier.
- Model file: `ml/models/baseline_video_classifier.json`
- Purpose: pipeline proof-of-concept, not final model.

Observed baseline result:

- Dataset size: **201**
- Test clips: **18**
- Correct clips: **7**
- Accuracy: **0.389**

This was enough to test the API/live-camera flow, but it produced false alarms such as low-confidence `bleeding_visible` when no bleeding was present.

In [ ]:
baseline_path = PROJECT_ROOT / "ml" / "models" / "baseline_video_classifier.json"
baseline = json.loads(baseline_path.read_text(encoding="utf-8"))
print("Model type:", baseline["model_type"])
print("Feature version:", baseline["feature_version"])
print("Dataset size:", baseline["dataset_size"])
print("Accuracy:", baseline["evaluation"]["accuracy"])
print("Test count:", baseline["evaluation"]["test_count"])
print("Correct count:", baseline["evaluation"].get("correct_count"))

## 5. Pretrained Video Model Experiments

Pretrained video model path:

- Backbone: TorchVision `r3d_18`
- Pretraining: general action-recognition video data
- Project use: replace final classification head and fine-tune on simulated clips

### 18-label detailed classifier

The 18-label classifier performed poorly:

| Epoch | Train Loss | Test Accuracy |
|---:|---:|---:|
| 1 | 3.4302 | 0.091 |
| 2 | 3.2378 | 0.061 |
| 3 | 3.2155 | 0.000 |

Conclusion: 18 detailed labels are too sparse and visually overlapping for the current dataset.

## 6. Grouped Classifier Experiments

### Grouped 3-epoch frozen-backbone run

| Epoch | Train Loss | Test Accuracy |
|---:|---:|---:|
| 1 | 1.6751 | 0.256 |
| 2 | 1.6289 | 0.308 |
| 3 | 1.6582 | 0.410 |

This was a major improvement over 18 detailed labels.

### Grouped 10-epoch frozen-backbone run

| Epoch | Train Loss | Test Accuracy |
|---:|---:|---:|
| 1 | 1.7411 | 0.333 |
| 2 | 1.7277 | 0.128 |
| 3 | 1.6424 | 0.231 |
| 4 | 1.6962 | 0.282 |
| 5 | 1.6944 | 0.359 |
| 6 | 1.6347 | 0.308 |
| 7 | 1.7057 | 0.333 |
| 8 | 1.6431 | 0.256 |
| 9 | 1.6963 | 0.231 |
| 10 | 1.6909 | 0.282 |

Conclusion: more frozen epochs did not reliably improve validation accuracy.

### Grouped unfrozen-backbone run, learning rate 0.0001

| Epoch | Train Loss | Test Accuracy |
|---:|---:|---:|
| 1 | 1.5224 | 0.436 |
| 2 | 1.2932 | 0.487 |
| 3 | 0.9746 | 0.385 |
| 4 | 0.6572 | 0.308 |
| 5 | 0.4072 | 0.231 |

Best result so far: **0.487** at epoch 2.

This shows overfitting: training loss kept decreasing, but test accuracy got worse after epoch 2.

### Grouped unfrozen-backbone run, learning rate 0.00005, early stopping

| Epoch | Train Loss | Test Accuracy |
|---:|---:|---:|
| 1 | 1.5427 | 0.308 |
| 2 | 1.4470 | 0.385 |
| 3 | 1.3746 | 0.462 |
| 4 | 1.2735 | 0.333 |
| 5 | 1.1293 | 0.333 |

Best result from this run: **0.462** at epoch 3.

Conclusion: lower learning rate reduced the collapse, but did not beat the previous best of 0.487.

In [ ]:
import torch

checkpoint_path = PROJECT_ROOT / "ml" / "models" / "video_action_grouped_classifier.pt"
if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    print("Checkpoint:", checkpoint_path)
    print("Label mode:", checkpoint.get("label_mode"))
    print("Labels:", checkpoint.get("labels"))
    print("Train count:", checkpoint.get("train_count"))
    print("Test count:", checkpoint.get("test_count"))
    print("Best epoch:", checkpoint.get("best_epoch"))
    print("Best accuracy:", checkpoint.get("best_accuracy"))
    print("History:")
    for row in checkpoint.get("history", []):
        print(row)
else:
    print("Checkpoint not found:", checkpoint_path)

## 7. Why Accuracy Is Still Low

Possible causes:

1. **Dataset size is small for video fine-tuning**  
   201 videos is useful for a prototype, but small for robust video action recognition.

2. **18 detailed labels are sparse**  
   Some labels have only 6-8 clips.

3. **Grouped labels still overlap visually**  
   For example, pacing, walking, standing, agitation, and normal movement can look similar.

4. **Simulated videos may have shared background, clothing, lighting, and actor style**  
   The model may learn scene or actor details instead of behavior.

5. **Pretraining domain mismatch**  
   `r3d_18` is pretrained on general action videos, not clinical observation or safety-monitoring scenes.

6. **Overfitting**  
   In unfrozen training, loss decreased while validation accuracy fell. This means the model memorized training clips.

7. **Single video-level labels are coarse**  
   A 40-second video may include multiple actions, but it receives one label.

8. **Frame sampling may miss the important moment**  
   The action may occur briefly, while sampled frames may capture irrelevant parts.

9. **Class imbalance**  
   Grouped labels are better than detailed labels, but `movement_agitation` still has only 15 videos.

10. **Some target events require object or pose reasoning, not pure action classification**  
    Sharp object risk should be handled by object detection. Pacing/inactivity should use pose/movement signals.

## 8. Why Pretrained Models Still Need Adaptation

Pretrained models help because they already understand general visual features, but they do not automatically understand this project's clinical labels.

Examples:

- YOLO can detect common objects, but may not know every clinically relevant dangerous object unless fine-tuned.
- MediaPipe can track body landmarks, but it does not directly label `pacing`, `head_banging`, or `prolonged_inactivity` without additional logic.
- `r3d_18` can understand general action patterns, but not specialized clinical-observation risk groups without fine-tuning.

Therefore, the best near-term system should not depend on a single video classifier. It should combine pretrained detectors with rules and structured classification.

## 9. Recommended Hybrid Architecture

Recommended Version 1 architecture:

```text
Live camera
→ YOLO object/person detection
→ MediaPipe pose and movement analysis
→ structured feature extraction
→ rule-based safety risk engine
→ small risk classifier over structured signals
→ LLM observation-note draft
→ staff review and alarm workflow
```

In this architecture, the model does not need to learn everything from raw pixels. The classifier can use structured signals such as:

```text
person_detected=true
sharp_object_detected=true
pose_motion=high
posture_change=large
time_inactive_seconds=120
multiple_people_close=true
risk_group=object_self_harm_risk
```

This is more realistic than training a full behavior classifier from the current dataset alone.

## 10. Current Best Model Position

Current best observed grouped video classifier result:

- Best accuracy observed: **0.487**
- Current checkpoint after latest low-learning-rate run: **0.462** unless restored/retrained
- Strong conclusion: video classifier alone is not sufficient for reliable clinical safety detection.

This is not a failure of using pretrained models. It means the problem requires a hybrid safety system rather than a single end-to-end video classifier.

## 11. Next Engineering Steps

Recommended next steps:

1. Add a proper confusion-matrix evaluation script for the grouped checkpoint.
2. Build a structured risk engine that combines YOLO and MediaPipe outputs.
3. Add a small structured-feature classifier for risk groups.
4. Use the LLM to draft observation notes from structured events, not directly from raw video.
5. Use the simulated dataset for validation, threshold tuning, and demonstration.

The dataset remains useful, but it should not be the only intelligence source in the system.

## 11. YOLO and MediaPipe Four-Video Test

This section tests the current pretrained-analyzer path on four simulated videos. These videos are used here as validation examples, not to train YOLO or MediaPipe. YOLO supplies object cues; MediaPipe supplies body landmark and movement cues. The combined clinical pipeline can use both at the same time, for example a person pacing while also holding or wearing an object that creates a risk cue.

| Video label | Video path | YOLO result | Dangerous objects | Clinical object cues | Pose coverage | Mean motion | Max motion | Posture change | Interpretation |
|---|---|---|---|---|---:|---:|---:|---:|---|
| pacing | `dataset/raw/pacing/pacing_001.mp4` | person, bottle, cup, couch; no dangerous object cue | `[]` | `[]` | 0.75 | 0.3279 | 1.2566 | 0.0414 | MediaPipe detected high movement; YOLO did not detect a dangerous object. |
| sharp_object_detected | `dataset/raw/sharp_object_detected/sharp_object_detected_001.mp4` | person, tie | `[]` | `[possible_ligature_cue]` | 0.00 | 0.0000 | 0.0000 | 0.0000 | YOLO produced a clinical caution cue; MediaPipe did not get usable pose landmarks in sampled frames. |
| fall | `dataset/raw/fall/fall_001.mp4` | person, car, chair, fire hydrant, suitcase; no dangerous object cue | `[]` | `[]` | 0.75 | 0.0559 | 0.1353 | 0.0164 | MediaPipe detected moderate movement; YOLO found general objects only. |
| prolonged_inactivity | `dataset/raw/prolonged_inactivity/prolonged_inactivity_001.mp4` | person, couch; no dangerous object cue | `[]` | `[]` | 1.00 | 0.0411 | 0.1334 | 0.0358 | MediaPipe tracked the body across sampled frames; YOLO found person/couch only. |


In [ ]:
from IPython.display import Video, display
from pathlib import Path

video_examples = [
    ("pacing", PROJECT_ROOT / "dataset/raw/pacing/pacing_001.mp4"),
    ("sharp_object_detected", PROJECT_ROOT / "dataset/raw/sharp_object_detected/sharp_object_detected_001.mp4"),
    ("fall", PROJECT_ROOT / "dataset/raw/fall/fall_001.mp4"),
    ("prolonged_inactivity", PROJECT_ROOT / "dataset/raw/prolonged_inactivity/prolonged_inactivity_001.mp4"),
]

for label, path in video_examples:
    print(label, path)
    display(Video(str(path), embed=True, width=480))


In [ ]:
from ml.inference.object_detector import (
    PretrainedObjectDetector,
    clinical_object_cues_from_detections,
    dangerous_objects_from_detections,
)
from ml.inference.pose_detector import MediaPipePoseMovementAnalyzer

object_detector = PretrainedObjectDetector()
pose_analyzer = MediaPipePoseMovementAnalyzer()

model_test_rows = []
for label, path in video_examples:
    detections = object_detector.detect_video(path, sample_count=4)
    pose = pose_analyzer.analyze_video(path, sample_count=4)
    model_test_rows.append(
        {
            "label": label,
            "video_path": str(path.relative_to(PROJECT_ROOT)),
            "yolo_labels": sorted({d.label for d in detections}),
            "dangerous_objects": dangerous_objects_from_detections(detections),
            "clinical_object_cues": clinical_object_cues_from_detections(detections),
            "pose_coverage": round(pose.pose_coverage, 3),
            "mean_motion": round(pose.mean_motion, 4),
            "max_motion": round(pose.max_motion, 4),
            "posture_change": round(pose.posture_change, 4),
        }
    )

model_test_rows


[
  {
    "label": "pacing",
    "video_path": "dataset/raw/pacing/pacing_001.mp4",
    "yolo_summary": "person, bottle, cup, couch; no dangerous object cue",
    "dangerous_objects": "[]",
    "clinical_object_cues": "[]",
    "pose_coverage": 0.75,
    "mean_motion": 0.3279,
    "max_motion": 1.2566,
    "posture_change": 0.0414
  },
  {
    "label": "sharp_object_detected",
    "video_path": "dataset/raw/sharp_object_detected/sharp_object_detected_001.mp4",
    "yolo_summary": "person, tie",
    "dangerous_objects": "[]",
    "clinical_object_cues": "[possible_ligature_cue]",
    "pose_coverage": 0.0,
    "mean_motion": 0.0,
    "max_motion": 0.0,
    "posture_change": 0.0
  },
  {
    "label": "fall",
    "video_path": "dataset/raw/fall/fall_001.mp4",
    "yolo_summary": "person, car, chair, fire hydrant, suitcase; no dangerous object cue",
    "dangerous_objects": "[]",
    "clinical_object_cues": "[]",
    "pose_coverage": 0.75,
    "mean_motion": 0.0559,
    "max_motion": 0.1353

In [ ]:
from ml.inference.clinical_pipeline import PretrainedClinicalObservationPipeline

pipeline = PretrainedClinicalObservationPipeline(
    baseline_model_path=PROJECT_ROOT / "ml/models/baseline_video_classifier.json",
    action_model_path=PROJECT_ROOT / "ml/models/video_action_grouped_classifier.pt",
)

combined_results = []
for label, path in video_examples:
    prediction = pipeline.predict(path)
    combined_results.append(
        {
            "label": label,
            "predicted_behaviour": prediction.predicted_behaviour,
            "confidence": round(prediction.confidence, 3),
            "risk_group": prediction.risk_group,
            "risk_level": prediction.risk_level,
            "alarm_required": prediction.alarm_required,
            "risk_reasons": prediction.risk_reasons,
            "model_version": prediction.model_version,
        }
    )

combined_results


### Notes from this test

- YOLO and MediaPipe are pretrained analyzers. The simulated videos are used here as test examples, not as training data for those two models.
- YOLO is best for object cues such as person, knife, scissors, tie-like cue, couch, chair, bottle, and other COCO objects.
- MediaPipe is best for body landmarks and movement cues such as pacing, posture change, low movement, and high movement.
- One video can produce both object and movement signals; the risk engine now keeps multiple reasons so combined risk can be escalated.
- Some YOLO detections are general environmental objects and are not safety risks by themselves.
